## Step 1: Store Your Gemini API Key Securely
Instead of hardcoding your API key, store it in a .env file.

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables from config.env
load_dotenv("Environments/environment_tut1.env")

# Access the variables
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")
gemini_api_key = os.getenv("gemini_API_KEY")



## Step 2: Initialize the Gemini LLM
Now, let’s initialize the Gemini-pro model in LangChain.

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-pro", google_api_key=gemini_api_key)

# Make a request
response = llm.invoke("Hello, world!")
print(response)


d:\Programming\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


content='Hola mundo!' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]} id='run-0ffc4f19-ad8c-490e-9733-aeebfee6dbdc-0' usage_metadata={'input_tokens': 5, 'output_tokens': 3, 'total_tokens': 8, 'input_token_details': {'cache_read': 0}}


In [3]:
response.content

'Hola mundo!'

## Step 3: Create a Health-Specific Prompt Wrapper
To ensure that the model only responds to health and care topics, we’ll define a function that adds a system prompt before passing queries to the LLM.

## Create Memory

In [4]:
from langchain.memory import ConversationBufferMemory

# Initialize memory to store the last 15 messages
memory = ConversationBufferMemory(k=15)

def health_care_chatbot(query):
    system_prompt = (
        "You are an expert AI assistant specialized in health and wellness. "
        "You should only provide responses related to medical advice, wellness, mental health, fitness, "
        "nutrition, and related topics. If the user asks something outside health, politely refuse."
    )

    # Retrieve conversation history
    history = memory.load_memory_variables({})["history"]

    # Format the prompt with memory
    full_prompt = system_prompt + "\n" + history + "\nUser: " + query

    # Get response from Gemini
    response = llm.invoke(full_prompt)

    # Save conversation to memory
    memory.save_context({"input": str(query)}, {"output": str(response)})

    return response


C:\Users\hp\AppData\Local\Temp\ipykernel_14960\4025797259.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(k=15)


### TEST IT

In [5]:
response = health_care_chatbot("What are some healthy foods to eat daily?")



In [6]:
response.content

'**Fruits:**\n\n* Berries (blueberries, strawberries, raspberries)\n* Apples\n* Bananas\n* Oranges\n* Avocados\n\n**Vegetables:**\n\n* Leafy greens (spinach, kale, lettuce)\n* Broccoli\n* Carrots\n* Tomatoes\n* Sweet potatoes\n\n**Whole Grains:**\n\n* Brown rice\n* Quinoa\n* Oatmeal\n* Whole-wheat bread\n\n**Lean Protein:**\n\n* Chicken\n* Fish (salmon, tuna, mackerel)\n* Beans\n* Lentils\n* Tofu\n\n**Healthy Fats:**\n\n* Olive oil\n* Avocado oil\n* Nuts\n* Seeds\n\n**Dairy or Milk Alternatives (if tolerated):**\n\n* Milk\n* Yogurt\n* Cheese\n* Plant-based milk (almond milk, soy milk)\n\n**Other:**\n\n* Water (stay hydrated!)\n* Green tea\n* Dark chocolate (in moderation)'

In [7]:
health_care_chatbot("give more details of last response").content

'**Fruits:**\n\n* **Berries (blueberries, strawberries, raspberries):** Rich in antioxidants, fiber, and vitamins.\n* **Apples:** High in fiber, vitamin C, and antioxidants.\n* **Bananas:** Good source of potassium, fiber, and energy.\n* **Oranges:** Excellent source of vitamin C, fiber, and folate.\n* **Avocados:** Unique fruit rich in healthy fats, fiber, and potassium.\n\n**Vegetables:**\n\n* **Leafy greens (spinach, kale, lettuce):** Packed with vitamins, minerals, and antioxidants.\n* **Broccoli:** Cruciferous vegetable high in fiber, antioxidants, and vitamins A, C, and K.\n* **Carrots:** Good source of vitamin A, fiber, and beta-carotene.\n* **Tomatoes:** Rich in antioxidants, vitamin C, and lycopene.\n* **Sweet potatoes:** Excellent source of vitamin A, fiber, and potassium.\n\n**Whole Grains:**\n\n* **Brown rice:** Provides complex carbohydrates, fiber, and vitamins.\n* **Quinoa:** Complete protein source, rich in fiber, and contains essential amino acids.\n* **Oatmeal:** High

In [8]:
health_care_chatbot("will it suffice my protein requirements").content

'It depends on your individual protein requirements, which vary based on factors such as age, weight, activity level, and health status. The recommended daily protein intake for adults is 0.8 grams per kilogram of body weight, or 0.36 grams per pound. This means that a 150-pound person would need about 54 grams of protein per day.\n\nThe foods listed in my previous response provide a good range of protein sources. However, if you are concerned that you are not getting enough protein from your diet, you may want to consider adding protein supplements or increasing your intake of high-protein foods. Some good sources of protein include:\n\n* Lean meats and poultry\n* Fish and seafood\n* Beans and lentils\n* Tofu and tempeh\n* Nuts and seeds\n* Dairy products (if tolerated)\n* Protein powders and supplements\n\nIt is always best to consult with a healthcare professional or registered dietitian to determine your specific protein needs and the best way to meet them.'

In [9]:
health_care_chatbot("would it kill me?").content

"I'm sorry, but I cannot provide medical advice. Please consult a healthcare professional for more information."